# Meijer API Exception Classes

This notebook demonstrates the custom exception classes available in the Meijer API client using real API calls.

## Overview

The `exceptions.py` module contains custom exception classes that provide meaningful error handling for different types of API failures.

## Exception Hierarchy

```
MeijerError (Base Exception)
├── MeijerAuthenticationError
├── MeijerAPIError
├── MeijerRateLimitError
├── CartError
└── FeedbackError
```

## Setup

First, let's import the necessary modules and create a real Meijer client:


In [ ]:
# Import the exception classes and client
from meijer.exceptions import (
    MeijerError,
    MeijerAuthenticationError,
    MeijerAPIError,
    MeijerRateLimitError,
    CartError,
    FeedbackError
)
from meijer.client import Meijer

print("✅ All exception classes imported successfully!")

# Create a real Meijer client
try:
    client = Meijer()
    print("✅ Meijer client created successfully!")
    print(f"Authentication status: {client.auth_status}")
    print(f"Is authenticated: {client.is_authenticated()}")
except Exception as e:
    print(f"❌ Failed to create Meijer client: {e}")
    client = None

## MeijerError - Base Exception

The `MeijerError` class is the base exception for all Meijer API errors.

### Real API Usage


In [ ]:
# Test MeijerError with real client
try:
    raise MeijerError("A generic Meijer API error occurred")
except MeijerError as e:
    print(f"✅ Caught MeijerError: {e}")
    print(f"Error type: {type(e).__name__}")

# Real client error handling
if client:
    try:
        if not client.is_authenticated():
            print("Client not authenticated - this is expected")
        else:
            # Test API call that might fail
            shopping_lists = client.get_shopping_lists()
            print(f"Retrieved {len(shopping_lists)} shopping lists")
    except Exception as e:
        print(f"API call failed: {type(e).__name__}: {e}")
else:
    print("❌ Client not available")

## MeijerAuthenticationError

Raised when authentication-related issues occur.

### Real API Usage


In [ ]:
# Test MeijerAuthenticationError
try:
    raise MeijerAuthenticationError("Authentication failed")
except MeijerAuthenticationError as e:
    print(f"✅ Caught MeijerAuthenticationError: {e}")

# Real authentication testing
if client:
    print(f"Current auth status: {client.auth_status}")
    
    if client.auth_status == "unauthenticated":
        try:
            shopping_lists = client.get_shopping_lists()
            print("Unexpectedly succeeded")
        except MeijerAuthenticationError as e:
            print(f"✅ Expected auth error: {e}")
        except Exception as e:
            print(f"Different error: {type(e).__name__}: {e}")
else:
    print("❌ Client not available")

## MeijerAPIError

Raised when general API issues occur.

### Real API Usage


In [ ]:
# Test MeijerAPIError
try:
    raise MeijerAPIError("API call failed")
except MeijerAPIError as e:
    print(f"✅ Caught MeijerAPIError: {e}")

# Real API testing
if client and client.is_authenticated():
    try:
        # Test with invalid parameters
        stores = client.get_stores(zip_code="invalid")
        print(f"Found {len(stores)} stores")
    except MeijerAPIError as e:
        print(f"✅ API error: {e}")
    except Exception as e:
        print(f"Different error: {type(e).__name__}: {e}")
else:
    print("❌ Client not authenticated")

## MeijerRateLimitError

Raised when API rate limits are exceeded.

### Real API Usage


In [ ]:
# Test MeijerRateLimitError
try:
    raise MeijerRateLimitError("Rate limit exceeded")
except MeijerRateLimitError as e:
    print(f"✅ Caught MeijerRateLimitError: {e}")

# Real rate limiting demo
if client and client.is_authenticated():
    print("Testing rapid API calls...")
    import time
    
    for i in range(3):
        try:
            stores = client.get_stores(zip_code="49508", limit=1)
            print(f"Call {i+1}: Found {len(stores)} stores")
            time.sleep(0.1)  # Be respectful
        except MeijerRateLimitError as e:
            print(f"⏱️ Rate limit hit: {e}")
        except Exception as e:
            print(f"Error: {type(e).__name__}: {e}")
else:
    print("❌ Client not authenticated")

## CartError

Raised when cart operations fail.

### Real API Usage


In [ ]:
# Test CartError
try:
    raise CartError("Failed to add item to cart")
except CartError as e:
    print(f"✅ Caught CartError: {e}")

# Real cart testing
if client and client.is_authenticated():
    try:
        # Try to add invalid item
        result = client.add_to_cart(product_id="invalid", quantity=1)
        print("Unexpectedly succeeded")
    except CartError as e:
        print(f"✅ Expected cart error: {e}")
    except Exception as e:
        print(f"Different error: {type(e).__name__}: {e}")
else:
    print("❌ Client not authenticated")

## FeedbackError

Raised when feedback operations fail.

### Real API Usage


In [ ]:
# Test FeedbackError
try:
    raise FeedbackError("Failed to submit feedback")
except FeedbackError as e:
    print(f"✅ Caught FeedbackError: {e}")

# Real feedback testing
if client and client.is_authenticated():
    try:
        # Try invalid feedback
        result = client.submit_feedback(
            product_id="invalid",
            rating=6,  # Invalid rating
            comment="Test"
        )
        print("Unexpectedly succeeded")
    except FeedbackError as e:
        print(f"✅ Expected feedback error: {e}")
    except Exception as e:
        print(f"Different error: {type(e).__name__}: {e}")
else:
    print("❌ Client not authenticated")

## Advanced Error Handling

Comprehensive error handling with real API calls.


In [ ]:
# Advanced error handling
def handle_meijer_operation(operation_name, operation_func, *args, **kwargs):
    """Generic error handler for Meijer operations."""
    print(f"\nExecuting: {operation_name}")
    
    try:
        result = operation_func(*args, **kwargs)
        print(f"✅ {operation_name} succeeded")
        return result
        
    except MeijerAuthenticationError as e:
        print(f"🔐 Authentication error: {e}")
        return None
        
    except MeijerRateLimitError as e:
        print(f"⏱️ Rate limit error: {e}")
        return None
        
    except CartError as e:
        print(f"🛒 Cart error: {e}")
        return None
        
    except FeedbackError as e:
        print(f"💬 Feedback error: {e}")
        return None
        
    except MeijerAPIError as e:
        print(f"🌐 API error: {e}")
        return None
        
    except MeijerError as e:
        print(f"❌ General Meijer error: {e}")
        return None
        
    except Exception as e:
        print(f"💥 Unexpected error: {type(e).__name__}: {e}")
        return None

# Test with real operations
if client and client.is_authenticated():
    # Test store search
    handle_meijer_operation(
        "Store Search",
        client.get_stores,
        zip_code="49508",
        limit=3
    )
    
    # Test product search
    handle_meijer_operation(
        "Product Search", 
        client.search_products,
        "milk",
        limit=5
    )
else:
    print("❌ Client not authenticated for advanced demo")

## Summary

This notebook demonstrated all exception classes using **real API calls**:

- **MeijerError**: Base exception for all Meijer errors
- **MeijerAuthenticationError**: Authentication issues
- **MeijerAPIError**: General API problems  
- **MeijerRateLimitError**: Rate limiting issues
- **CartError**: Shopping cart failures
- **FeedbackError**: Feedback operation failures

### Best Practices

1. Always catch specific exception types first
2. Implement proper logging for all errors
3. Provide user-friendly error messages
4. Use exponential backoff for rate limiting
5. Handle authentication gracefully

All examples use actual Meijer API endpoints instead of mocked data.
